# 🌿 Workshop: Serra Domotica ESP32 — Generazione STL con AI
### Dal prompt al pezzo stampato: genera, slicia, correggi, ristampa



## 🗓️ Programma del workshop

| Blocco | Obiettivo |
|--------|-----------|
| **0. Setup** | Ambiente Colab pronto |
| **1. Base** | Genera il pannello base con Gemini, esporta, guarda nello slicer |
| **2. Body** | Genera il telaio della serra (4 pannelli per faccia), esporta, guarda nello slicer |
| **3. Tetto** | Genera la capriata del tetto a falda, gestisci più travi che si incontrano |
| **4. Finestre PETG** | Telaietto con battuta per UN pannello + 1 lastra di prova da stampare |
| **4bis. Supporto Ventola** | Placca con foro flusso, feritoie e fori vite (40 mm) |
| **5. 🛠️ Esercitazione libera** | Ognuno disegna e stampa la propria decorazione |
| **6. Feedback** | Mostra il pezzo, commenti live |
| **7. Chiusura** | Download file, risorse, prossimi passi |

> 🎯 **Come lavoriamo**: per ogni pezzo, copi il **Prompt Gemini** indicato nel blocco,
> lo dai a **Gemini**, e il codice Python che ti restituisce lo incolli nella cella
> `# TODO` subito sotto. Esegui la cella: si genera il file `.stl`. Poi lo guardi nella
> preview 3D e lo controlli nello **slicer** (PrusaSlicer / Bambu Studio).
>
> Se qualcosa non va — dimensioni sbagliate, mesh che lo slicer segnala come
> "non chiusa"/"non manifold", o un pezzo (trave/montante) che sembra mancare — non è
> un tuo errore: è normale, fa parte del metodo. Torna su Gemini, incolla l'errore o la
> descrizione del problema, e chiedi una correzione. Poi ri-esegui la cella e riprova.
> Questo avanti-e-indietro è il cuore del workshop, non un imprevisto.


### ✅ Checklist pre-workshop

- [ ] Account Google attivo (per Colab)
- [ ] Accesso a Gemini (browser o app)
- [ ] Slicer della propria stampante installato e pronto (PrusaSlicer / Bambu Studio / altro)
- [ ] Una stampante 3D collegata (anche una sola condivisa per il gruppo va bene)

### 🧰 Cosa serve durante il workshop
- Laptop con browser (Colab gira nel cloud, nessuna installazione Python locale)
- Una seconda finestra/tab con Gemini aperto
- Lo slicer aperto e pronto a importare file STL


---
# 0. Setup
Installiamo le librerie minime. Per **Base** e per la **Decorazione libera** lavoriamo
direttamente con triangoli e mesh, lo stesso linguaggio che usa un file STL — più diretto
da far scrivere all'AI e più facile da capire leggendo il codice che genera.

Per **Body, Roof, Telaietto PETG e Supporto Ventola** usiamo invece `trimesh` con
operazioni booleane (`manifold3d`): si parte da forme semplici (box, cilindri) e si
sottraggono o uniscono fra loro, invece di scrivere a mano ogni singolo triangolo del
taglio. Quando un pezzo è fatto di tante travi, fori o feritoie diverse, un'operazione
booleana resta leggibile; uno script di soli triangoli manuali inizia a diventare fragile
— lo vedremo chiaramente nel Blocco 2 (telaio del Body) e nel Blocco 3 (capriata del tetto).


In [ ]:
# Installazione librerie (circa 30-40 secondi)
!pip install numpy numpy-stl trimesh manifold3d matplotlib --quiet

import numpy as np
from stl import mesh
import trimesh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import os

print('✅ Ambiente pronto')
print(f'NumPy: {np.__version__} | Trimesh: {trimesh.__version__}')


In [ ]:
def visualizza_stl(filepath, title='', color='#2e7d52'):
    """Mostra una preview 3D veloce di un file STL, prima di mandarlo allo slicer."""
    m = mesh.Mesh.from_file(filepath)
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(111, projection='3d')
    poly = Poly3DCollection(m.vectors, alpha=0.4, facecolor=color, edgecolor='gray', linewidth=0.3)
    ax.add_collection3d(poly)
    v = m.vectors.reshape(-1,3)
    ax.set_xlim(v[:,0].min(), v[:,0].max())
    ax.set_ylim(v[:,1].min(), v[:,1].max())
    ax.set_zlim(v[:,2].min(), v[:,2].max())
    ax.set_xlabel('X (mm)'); ax.set_ylabel('Y (mm)'); ax.set_zlabel('Z (mm)')
    ax.set_title(title or filepath)
    plt.tight_layout()
    plt.show()

def info_stl(filepath):
    """Dimensioni essenziali — giusto per controllare al volo prima di scaricare."""
    m = mesh.Mesh.from_file(filepath)
    v = m.vectors.reshape(-1, 3)
    dx = v[:,0].max() - v[:,0].min()
    dy = v[:,1].max() - v[:,1].min()
    dz = v[:,2].max() - v[:,2].min()
    print(f'{filepath}: {dx:.1f} × {dy:.1f} × {dz:.1f} mm — {len(m.vectors)} triangoli')

print('✅ Funzioni di preview pronte')

---
# 1. Base

Il pannello di base della serra: un pannello con una finestra aperta nella parte alta.

### 📋 Prompt Gemini
```
Scrivi uno script Python che generi una mesh STL usando solo numpy e numpy-stl
(nessuna libreria CAD), costruendo i triangoli manualmente.

Il pezzo è un pannello rettangolare 49 × 151.3 × 151.6 mm (X × Y × Z), pieno,
con un'apertura rettangolare passante nella parte alta: la finestra è centrata
in Y tra 59 e 94 mm, e si apre da Z=145 mm fino alla sommità (Z=151.6 mm),
attraversando tutto lo spessore in X tranne 4 mm di parete per lato.

Costruisci la mesh come una lista di triangoli (facce esterne del solido più
le facce interne della finestra) e salva il risultato in 'Base.stl' con
mesh.Mesh, salvando in formato binario.
```

Incolla il codice che ottieni da Gemini nella cella sotto, eseguilo, e guarda il risultato.


In [ ]:
# ── TODO: incolla qui il codice generato da Gemini per Base.stl ──
# 1. Copia il prompt Gemini del blocco qui sopra
# 2. Portalo su Gemini, ottieni lo script Python
# 3. Incolla qui sotto il codice (sostituendo questo scaffold)
# 4. Esegui la cella: deve produrre il file 'Base.stl'
#
# Suggerimento sulla firma della funzione, se vuoi un punto di partenza:
# def crea_box_con_foro(dim, foro_y, foro_z, sp_x): ...

import numpy as np
from stl import mesh

# TODO: sostituisci questa riga con il codice di Gemini
raise NotImplementedError('Incolla qui il codice generato da Gemini per Base.stl')


In [ ]:
visualizza_stl('Base.stl', 'Base')

### 🖨️ Ora vai sullo slicer
Scarica `Base.stl` (cella di download nel Blocco 7, oppure trascina il file dalla barra
laterale dei file di Colab) e aprilo nel tuo slicer. Controlla:
- Le dimensioni sono quelle attese (49 × 151.3 × 151.6 mm)?
- La finestra è nella posizione giusta?
- La mesh è chiusa (lo slicer di solito segnala mesh non manifold/aperte)?

Se qualcosa non va, torna su Gemini, descrivi cosa hai visto nello slicer (es. "la mesh
ha un buco sul lato sinistro") e chiedi una correzione. Poi torna qui, aggiorna la cella
e ri-esegui.


---
# 2. Body

Il corpo principale della serra: uno **scheletro/telaio**, non un box a pareti piene.
Ogni faccia laterale è divisa in 4 pannelli quadrati vuoti da una croce di montanti
(verticale + orizzontale); top e bottom restano pieni. I pannelli vuoti ospiteranno poi,
come pezzi separati, le finestre PETG (Blocco 4) e il supporto ventola (Blocco 4bis).

### 📋 Prompt Gemini
```
Scrivi uno script Python che generi una mesh STL usando trimesh (non triangoli
manuali): parti da un box pieno 151.5 × 151.6 × 151.3 mm (X × Y × Z) e SOTTRAI
con operazioni booleane (engine 'manifold') le aperture per renderlo un telaio:

Su ciascuna delle 4 facce laterali (fronte Y=0, retro Y=Y, sinistra X=0, destra
X=X — non su top e bottom), taglia 4 finestre quadrate passanti, una per
quadrante, lasciando una cornice perimetrale di 7 mm e una croce centrale
(montante verticale + traversa orizzontale) sempre di 7 mm, che dividono ogni
faccia in 4 pannelli quadrati uguali.

Esporta il risultato in 'Body.stl'.
```

Incolla il codice nella cella sotto, oppure usa il punto di partenza già pronto.


In [ ]:
# ── TODO: incolla qui il codice generato da Gemini per Body.stl ──
# 1. Copia il prompt Gemini del blocco qui sopra
# 2. Portalo su Gemini, ottieni lo script Python
# 3. Incolla qui sotto il codice (sostituendo questo scaffold)
# 4. Esegui la cella: deve produrre il file 'Body.stl'
#
# Suggerimento sulla firma della funzione, se vuoi un punto di partenza:
# def crea_body_telaio(dim, spessore_telaio=7.0): ...  # usa trimesh + booleane

import numpy as np
from stl import mesh

# TODO: sostituisci questa riga con il codice di Gemini
raise NotImplementedError('Incolla qui il codice generato da Gemini per Body.stl')


In [ ]:
visualizza_stl('Body.stl', 'Body', color='#1565c0')

### 🖨️ Vai sullo slicer
Apri `Body.stl` nello slicer. Controlla volume e dimensioni nel pannello informazioni:
essendo costruito per operazioni booleane, dovrebbe risultare già una mesh chiusa, senza
avvisi di riparazione. Se qualcosa non corrisponde (pannelli storti, montanti mancanti),
torna su Colab e controlla i parametri `pannello_w`/`pannello_h`/`pannello_d`, o chiedi a
Gemini di correggere lo script descrivendo cosa hai visto.


---
### 🌀 Nota di design — la ventola si fa, ma più avanti

Uno dei 4 pannelli vuoti della faccia X0 (o X1) del Body — lo stesso tipo di vano
~65 × 65 mm che ospiterà anche un telaietto PETG sulle altre facce — è quello che
useremo per montare la **ventola di aerazione da 40 mm**. Anche il supporto ventola
lo generiamo con lo stesso metodo prompt → Colab → slicer — lo trovi nel Blocco 4bis,
dopo le finestre PETG.

L'unica parte che lasciamo al CAD vero (zoo.dev / Fusion 360) è **l'incastro di precisione**
fra il supporto stampato e il vano del Body: lì la tolleranza conta più della velocità.


---
# 3. Tetto

Anche il tetto è una **capriata** (traliccio), non un guscio a pareti piene: travi
sottili lungo gli spigoli del profilo a falda, più una capriata trasversale sulle due
testate e a metà lunghezza, con una diagonale di rinforzo per ciascuna capriata.
È il pezzo più delicato — non per le pareti piene di prima, ma perché ha più travi
che si incontrano nello stesso punto: un buon banco di prova per capire come l'AI
gestisce operazioni booleane multiple in sequenza.

### 📋 Prompt Gemini
```
Scrivi uno script Python che generi una mesh STL usando trimesh (non triangoli
manuali), costruendo il tetto per composizione (unione di travi), non come guscio.

Il pezzo è un tetto a una falda (shed roof): un profilo a triangolo rettangolo nel
piano YZ, estruso per 151.3 mm lungo l'asse X. Il triangolo ha vertici A=(2.5,2.5)
(angolo retto), B=(103.2,2.5) (base) e C=(103.2,103.2) (colmo), con spessore trave
di 5 mm — i vertici sono arretrati di metà spessore per restare dentro un bounding
box 151.3 × 105.7 × 105.7 mm una volta data sezione alle travi.

Costruisci:
1. 3 travi longitudinali (sezione 5×5mm) lungo X, una per vertice (A, B, C)
2. su 3 sezioni trasversali (X=2.5, X=75.65, X=148.8): i 3 lati del triangolo
   (A-B, B-C, C-A) come travi sottili sezione 5×5mm, più 1 diagonale di rinforzo
   dal punto medio dell'ipotenusa C-A al vertice B

Unisci tutte le travi con operazioni booleane (union, engine 'manifold').
Esporta il risultato in 'Roof.stl'.
```

Incolla il codice nella cella sotto.


In [ ]:
# ── TODO: incolla qui il codice generato da Gemini per Roof.stl ──
# 1. Copia il prompt Gemini del blocco qui sopra
# 2. Portalo su Gemini, ottieni lo script Python
# 3. Incolla qui sotto il codice (sostituendo questo scaffold)
# 4. Esegui la cella: deve produrre il file 'Roof.stl'
#
# Questo è il pezzo con più travi che si incontrano: se un'unione booleana fallisce silenziosamente, una trave può risultare assente anche se il codice non dà errori.
#
# Suggerimento sulla firma della funzione, se vuoi un punto di partenza:
# def crea_roof_capriata(lunghezza, larghezza, altezza, spessore_trave): ...  # trimesh + union

import numpy as np
from stl import mesh

# TODO: sostituisci questa riga con il codice di Gemini
raise NotImplementedError('Incolla qui il codice generato da Gemini per Roof.stl')


In [ ]:
visualizza_stl('Roof.stl', 'Roof', color='#c62828')

### 🖨️ Vai sullo slicer
Il tetto è il pezzo con più travi che si incrociano nello stesso punto: se qualche unione
booleana fallisce silenziosamente (capita quando due travi sono quasi collineari), una
trave può risultare assente nel risultato finale anche se il volume totale sembra giusto.
Apri `Roof.stl` nello slicer e conta a vista gli spigoli e le diagonali: se manca un pezzo,
torna su Gemini, descrivi cosa hai visto, e chiedi se due travi nel codice condividono
esattamente la stessa direzione (in tal caso vanno unite in un ordine diverso, o una delle
due va leggermente spostata).


---
# 4. Finestre in PETG traslucido

Ogni pannello vuoto del Body (telaio a croce, ~65 × 65 mm) ospita una finestra:
non un semplice rettangolo di PETG, ma un **telaietto** con una battuta (un gradino)
dove incastrare la lastra a filo con la faccia esterna — un pezzo a parte, stampato
separatamente e poi incastrato a frizione nel vano del telaio.

```
        ┌──────────────┐
        │░░░░░░░░░░░░░░│  ← telaietto, perimetro pieno
        │░░┌────────┐░░│  ← battuta: qui appoggia il vetro/PETG
        │░░│ FINESTRA │░░│  ← area ribassata (foro o sottile, a scelta)
        │░░└────────┘░░│
        │░░░░░░░░░░░░░░│
        └──────────────┘
```

### 🎯 Perché il PETG traslucido
- Buona trasparenza/translucenza per la luce delle piante
- Resistenza UV e umidità superiore al PLA
- Stessa dilatazione termica del resto della serra, se anche Body/Roof sono in PETG

Oggi se ne stampa **una** (di prova): lo stesso telaietto, ridimensionato sul singolo
pannello del telaio, si ripete identico **4 volte per faccia** una volta validato.

### 📋 Prompt Gemini — Telaietto PETG
```
Scrivi uno script Python che generi una mesh STL usando trimesh (non triangoli
manuali): un telaietto quadrato 64.9 × 64.9 × 4 mm (un po' più piccolo del vano
65.2 × 65.2 mm del Body, per entrare a frizione), con una battuta — un gradino
ribassato di 1.5 mm su un'area centrale 56.9 × 56.9 mm, lasciando un bordo pieno
di 4 mm su ogni lato. Costruiscilo per composizione: un box pieno meno un box più
piccolo sottratto dalla faccia superiore (operazione booleana, engine 'manifold').
Esporta in 'Telaietto_PETG.stl'.
```

### 📋 Prompt Gemini — lastra PETG di prova
```
Scrivi uno script Python che generi con trimesh una semplice lastra quadrata
piatta 56.6 × 56.6 × 2 mm (un po' più piccola dell'area della battuta, per
entrarci con un piccolo gioco). Esporta in 'Vetro_Prova.stl'.
```


In [ ]:
# ── TODO: incolla qui il codice generato da Gemini per Telaietto_PETG.stl ──
# 1. Copia il prompt Gemini del blocco qui sopra
# 2. Portalo su Gemini, ottieni lo script Python
# 3. Incolla qui sotto il codice (sostituendo questo scaffold)
# 4. Esegui la cella: deve produrre il file 'Telaietto_PETG.stl'
#
# Lo script deve restituire anche LATO_BATTUTA come variabile: ti servirà nella cella della lastra PETG subito dopo.
#
# Suggerimento sulla firma della funzione, se vuoi un punto di partenza:
# def crea_telaietto_petg(lato_pannello, spessore_telaietto, margine_battuta, profondita_battuta, tolleranza=0.3): ...  # trimesh + difference

import numpy as np
from stl import mesh

# TODO: sostituisci questa riga con il codice di Gemini
raise NotImplementedError('Incolla qui il codice generato da Gemini per Telaietto_PETG.stl')


In [ ]:
visualizza_stl('Telaietto_PETG.stl', 'Telaietto PETG', color='#00897b')


In [ ]:
# ── TODO: lastra PETG di prova ──
# Usa il prompt Gemini "lastra PETG di prova" qui sopra, oppure scrivi tu lo script:
# una semplice lastra quadrata piatta, dimensionata sulla battuta del telaietto
# (lato della battuta meno una piccola tolleranza di ~0.3 mm, in modo che entri
# senza forzare), spessore ~2 mm.
#
# Se hai già LATO_BATTUTA definito nel blocco precedente, riusalo qui invece di
# riscrivere i numeri a mano.

import trimesh

# TODO: sostituisci questa riga con il codice di Gemini (o il tuo)
raise NotImplementedError("Incolla qui il codice per generare Vetro_Prova.stl")


**Parametri di stampa per il PETG traslucido**:

| Parametro | Valore consigliato |
|-----------|---------------------|
| Temperatura ugello | 230–240 °C |
| Temperatura piatto | 70–80 °C |
| Velocità stampa | 30–40 mm/s |
| Altezza layer | 0.1–0.12 mm |
| Ventole | Minime/assenti |

### 🖨️ Vai sullo slicer
Manda `Vetro_Prova.stl` in stampa subito (insieme al `Telaietto_PETG.stl`): lavoreranno
in sottofondo mentre passiamo all'esercitazione libera. Una volta validato l'incastro
telaietto-vano e telaietto-lastra, lo stesso pezzo si ristampa **4 volte per faccia**
(16 in tutto, per le 4 facce del Body) — cambia solo la quantità, non il file.


---
# 4bis. Supporto Ventola

Il supporto per la ventola di aerazione da 40 mm: una placca con foro centrale per
il flusso d'aria, feritoie radiali (stile griglia di ventilazione) e i 4 fori per
le viti della ventola (passo 32 mm, lo standard per le ventole 40×40 mm).

### 📋 Prompt Gemini
```
Scrivi uno script Python che generi una mesh STL per una placca quadrata 50 × 50 ×
4 mm, usando trimesh (non triangoli manuali): parti da un box pieno e SOTTRAI con
operazioni booleane (engine 'manifold'):

1. un cilindro passante al centro, diametro 30 mm (foro per il flusso d'aria)
2. 4 cilindri passanti, diametro 4 mm, ai vertici di un quadrato di lato 32 mm
   centrato sulla placca (fori per le viti della ventola 40 mm)
3. 10 feritoie radiali passanti (piccoli box 4 × 2 mm, ruotati), distribuite ad
   anello a raggio 19 mm dal centro, per l'aerazione

Esporta il risultato in 'Supporto_Ventola.stl'.
```

Incolla il codice nella cella sotto, oppure usa il punto di partenza già pronto.


In [ ]:
# ── TODO: incolla qui il codice generato da Gemini per Supporto_Ventola.stl ──
# 1. Copia il prompt Gemini del blocco qui sopra
# 2. Portalo su Gemini, ottieni lo script Python
# 3. Incolla qui sotto il codice (sostituendo questo scaffold)
# 4. Esegui la cella: deve produrre il file 'Supporto_Ventola.stl'
#
# Qui si lavora per composizione (trimesh + sottrazione booleana), non a triangoli manuali: parti da una placca piena e sottrai cilindri/box come 'utensili'.
#
# Suggerimento sulla firma della funzione, se vuoi un punto di partenza:
# def crea_supporto_ventola(lato, spessore, diametro_flusso, diametro_vite, passo_viti, n_feritoie, raggio_feritoie, larghezza_feritoia, lunghezza_feritoia): ...

import numpy as np
from stl import mesh

# TODO: sostituisci questa riga con il codice di Gemini
raise NotImplementedError('Incolla qui il codice generato da Gemini per Supporto_Ventola.stl')


In [ ]:
visualizza_stl('Supporto_Ventola.stl', 'Supporto Ventola', color='#ef6c00')


### 🖨️ Vai sullo slicer (e poi nota bene)
Apri `Supporto_Ventola.stl` nello slicer: dovrebbe risultare già una mesh chiusa, perché
questa volta i buchi sono stati generati per sottrazione booleana, non a triangoli scritti
a mano — meno probabilità di normali invertite o piani scoperti.

**Quello che lo script NON fa, e va fatto a parte in CAD:**
- L'incastro di precisione fra questo supporto e il vano ~65 × 65 mm del Body
  (tolleranza fine, eventuale battuta o linguetta di centraggio — stessa idea del
  Telaietto PETG del Blocco 4, ma qui serve anche reggere il peso della ventola)
- Eventuali nervature di rinforzo attorno ai fori vite, se in stampa risultano fragili
- L'adattamento del diametro/posizione dei fori se la vostra ventola da 40 mm ha un passo
  viti diverso da 32 mm (controllate il datasheet prima di stampare in serie)

Per questi tre punti: zoo.dev (text-to-CAD, parte veloce) o Fusion 360 (finiture e
tolleranze esatte) — stesso discorso del Blocco 13 ("Quando serve davvero il CAD").


---
# 5. 🛠️ Esercitazione libera — la tua decorazione

Mentre la lastra PETG di prova stampa, disegna una piccola decorazione per la facciata
della serra. Scrivi tu il prompt per Gemini, partendo dal template minimo qui sotto.

### 🎯 Vincoli (per restare stampabile in pochi minuti)
- Ingombro massimo: **60 × 60 × 10 mm**
- Spessore base: almeno 2 mm
- Nessuna parte sospesa senza supporto sotto i 45°

### 💡 Idee di partenza (scegli una o mescola)
- Una **forma/logo** stilizzato (foglia, goccia, sole, iniziali)
- Un **foro passacavo** per un sensore (Ø 4–10 mm)
- Un **rilievo** che sporge di 1–3 mm dalla base

### 📋 Prompt Gemini — punto di partenza libero
```
Scrivi uno script Python che generi con numpy e numpy-stl una mesh STL,
costruendo i triangoli manualmente (nessuna libreria CAD).

Il pezzo è una placca rettangolare 60 × 60 × 3 mm. Aggiungi [LA TUA IDEA: es.
"un rilievo a forma di goccia stilizzata, alto 2mm, leggermente decentrato
verso l'alto"]. Salva il risultato in 'MiaDecorazione.stl' con mesh.Mesh,
in formato binario.
```

Scrivi il tuo prompt, portalo su Gemini, e incolla il codice ottenuto nella cella sotto.


In [ ]:
# ── TODO: incolla qui il codice generato da Gemini per la tua decorazione ──
# Se vuoi un punto di partenza minimo da cui ripartire invece che da zero,
# puoi generare prima una semplice placca piena (senza decorazione) e poi
# chiedere a Gemini di aggiungerci sopra la tua idea.

import numpy as np
from stl import mesh

# TODO: sostituisci questa riga con il codice di Gemini
raise NotImplementedError("Incolla qui il codice generato da Gemini per MiaDecorazione.stl")


In [ ]:
visualizza_stl('MiaDecorazione.stl', 'La mia decorazione', color='#6a1b9a')

### 🖨️ Vai sullo slicer
Scarica il file ed esportalo nello slicer. Controlla che il pezzo sia chiuso e stampabile
senza troppi supporti. Se serve un aggiustamento, torna su Gemini, descrivi cosa cambiare
e ottieni il codice corretto. Quando il pezzo ti convince, avvia la stampa.


---
# 6. Feedback live

Ogni partecipante mostra il proprio pezzo (in stampa, stampato, o a schermo se non è ancora pronto).

### 📝 Checklist di feedback (3 domande)
1. **Cosa funziona bene** nella forma o nel dettaglio?
2. **Cosa cambieresti** in una v2 (proporzioni, posizione, profondità)?
3. **È stampabile** così com'è, o servirebbe un piccolo aggiustamento (supporti, spessore minimo)?

> 💡 I pezzi migliori del gruppo possono diventare la decorazione ufficiale della serra finale,
> combinata sulla faccia frontale del Body.

# 7. Chiusura, download e risorse

In [ ]:
# ── Download di tutti i file generati oggi ──
from google.colab import files

output_files = [
    'Base.stl',
    'Body.stl',
    'Roof.stl',
    'Telaietto_PETG.stl',
    'Vetro_Prova.stl',
    'Supporto_Ventola.stl',
    'MiaDecorazione.stl',
]

print('📦 Download...')
for f in output_files:
    if os.path.exists(f):
        files.download(f)
        print(f'  ⬇️  {f}')
    else:
        print(f'  ⚠️  {f} non trovato')


### 🔧 Troubleshooting rapido

| Problema | Dove guardare |
|----------|----------------|
| (Base/Decorazione) Lo slicer segnala mesh non chiusa | Manca una faccia nei triangoli —
controlla che ogni lato del solido sia coperto da `quad(...)` o `tri(...)` |
| (Base/Decorazione) Una faccia appare "al contrario" (nera/trasparente) | L'ordine dei
vertici nel triangolo è invertito — scambia due vertici nella chiamata `quad`/`tri` |
| (Body/Roof/Telaietto/Ventola) Un pezzo della geometria sembra mancante | Due forme quasi
collineari o coincidenti possono fondersi senza errori ma "sparire" nell'unione booleana —
controlla l'ordine delle `union()`/`difference()` o sposta leggermente una delle due |
| Lastra PETG non entra nella battuta | Aumenta `tolleranza` nel Telaietto, verifica
`RABBET_DEPTH ≥ VETRO_SP` |
| `difference(...)`/`union(..., engine='manifold')` lancia un errore | Manca `manifold3d`:
ri-esegui la cella di Setup, o installa con `!pip install manifold3d --quiet` |
| La tua decorazione ha parti sospese | Aggiungi uno spessore di base sotto, o stampa con
supporti |

### 🛠️ Software utili dopo il workshop
- **PrusaSlicer / Bambu Studio** — verifica stampabilità, repair mesh, profili PETG
- **MeshLab** — ispezione e riparazione mesh più avanzata
- **Blender** — editing manuale, se vuoi smussare o rifinire a mano

### 🌿 Per arrivare a tutti i 16 telaietti PETG a casa
Lo stesso `Telaietto_PETG.stl` (e relativa `Vetro_Prova.stl`) si ristampa identico per
tutti i pannelli vuoti delle 4 facce del Body (4 pannelli × 4 facce = 16 telaietti in
tutto, meno quello/i occupati dal Supporto Ventola): cambia solo la quantità in stampa,
non il file. Se preferite più varietà, basta cambiare `RABBET_DEPTH` o `VETRO_SP` tra un
lotto e l'altro per confrontare risultati diversi.

### 🌬️ Per portare il Supporto Ventola dal Colab al pezzo vero
Questo STL è la base parametrica, non il pezzo finito: prima di stamparlo in via definitiva,
importatelo in zoo.dev o Fusion 360 insieme al Body (o solo al pannello ~65 × 65 mm di
riferimento) per disegnare l'incastro di precisione — tolleranza fine, eventuale battuta o
linguetta di centraggio — e per verificare passo viti e diametro contro il datasheet della
vostra ventola, se diversi da 32 mm / Ø 40 mm.

### 🌱 Prossimi passi per la serra domotica
1. **Sensori**: alloggio DHT22 (temp/umidità), sensore suolo capacitivo
2. **Elettronica**: slot ESP32 DevKit (54×28 mm), foro USB-C
3. **Finestre PETG**: incollaggio dei 16 telaietti con cianoacrilico compatibile o
   silicone trasparente
4. **Ventola**: rifinitura dell'incastro in CAD, poi assemblaggio sul pannello del Body
5. **Assemblaggio**: clip di giunzione tra Base + Body + Roof
6. **Firmware**: ESPHome o MicroPython → Home Assistant

Grazie per aver partecipato! 🌿
